In [4]:
# Libraries Loading

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import gc
import golois
import sys

print("Python version:", sys.version.split()[0])
print("TensorFlow version:", tf.__version__)

Python version: 3.9.21
TensorFlow version: 2.15.0


In [5]:
# Configuration

planes = 31
moves = 361
N = 10000

# Générattion de 10000 plateaux de jeu avec des données aléatoires
# 19x19x31
input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

# La policy (quel coup jouer) est un vecteur de 361 valeurs de 0 à 360
# on le convertit en un vecteur de 361 valeurs de 0 à 1
policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

# La value la policy (évaluation de la position)
# La value est un vecteur de 0 ou 1
value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

# Donnée de fin de partie par case 
end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

# Les end groups sont des groupes de pierres qui sont mortes
# ou vivantes à la fin de la partie
groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

# Modèle AlphaGo Zero
# Ce modèle est inspiré de l'architecture AlphaGo Zero

filter = 64

def build_alphago_model(input_shape=(19, 19, 31), filters=32, l2_reg=1e-4):
    input_layer = layers.Input(shape=input_shape)

    # 🔁 Blocs convolutionnels
    x = layers.Conv2D(filters, 3, padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(l2_reg))(input_layer)
    
    for _ in range(10):
        x = layers.Conv2D(filters, 3, padding='same', activation='relu',
                          kernel_regularizer=regularizers.l2(l2_reg))(x)
        x = layers.MaxPooling2D(pool_size=(2, 2), strides=2, padding='same')(x)

    # 🎯 Policy head
    policy_conv = layers.Conv2D(4, 1, activation='relu',
                                kernel_regularizer=regularizers.l2(l2_reg))(x)
    policy_gap = layers.GlobalAveragePooling2D()(policy_conv)
    policy_dense = layers.Dense(128, activation='relu')(policy_gap)
    policy_output = layers.Dense(361, activation='softmax', name='policy')(policy_dense)

    # 💡 Value head
    value_conv = layers.Conv2D(2, 1, activation='relu',
                               kernel_regularizer=regularizers.l2(l2_reg))(x)
    value_gap = layers.GlobalAveragePooling2D()(value_conv)
    value_dense = layers.Dense(64, activation='relu',
                               kernel_regularizer=regularizers.l2(l2_reg))(value_gap)
    value_output = layers.Dense(1, activation='sigmoid', name='value')(value_dense)

    model = models.Model(inputs=input_layer, outputs=[policy_output, value_output])
    model.summary()
    return model

model = build_alphago_model(input_shape=(19, 19, 31), filters=128, l2_reg=1e-4)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 19, 19, 31)]         0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 19, 19, 128)          35840     ['input_1[0][0]']             
                                                                                                  
 conv2d_1 (Conv2D)           (None, 19, 19, 128)          147584    ['conv2d[0][0]']              
                                                                                                  
 max_pooling2d (MaxPooling2  (None, 10, 10, 128)          0         ['conv2d_1[0][0]']            
 D)                                                                                           

2025-04-02 16:23:39.657289: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-04-02 16:23:39.657319: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-04-02 16:23:39.657327: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2025-04-02 16:23:39.657386: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-02 16:23:39.657612: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
# Train the Model
epochs = 10
batch = 128

to_train = True

if to_train:

    print("getValidation", flush=True)
    golois.getValidation(input_data, policy, value, end)

    sgd = keras.optimizers.SGD(learning_rate=0.1, momentum=0.9, nesterov=True)
    model.compile(
        optimizer=sgd,
        loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mse'}
    )

    # 🔁 Accumulateur des courbes
    full_history = {
        'loss': [],
        'policy_loss': [],
        'value_loss': [],
        'policy_categorical_accuracy': [],
        'value_mse': []
    }

    for i in range(1, epochs + 1):
        print('epoch', i)
        golois.getBatch(input_data, policy, value, end, groups, i * N)

        history = model.fit(
            input_data,
            {'policy': policy, 'value': value},
            epochs=1,
            batch_size=batch,
            verbose=1
        )

        for key in full_history:
            if key in history.history:
                full_history[key].append(history.history[key][0])

        if i % 5 == 0:
            gc.collect()

        if i % epochs == 0:
            golois.getValidation(input_data, policy, value, end)
            val = model.evaluate(input_data, [policy, value], verbose=1, batch_size=batch)
            print("val =", val)
            model.save('test.h5')



getValidation


r.shape = (10000, 19, 19, 31)
nbExamples = 10000
nbPositionsSGF = 102208897
nbPositionsSGF = 102208897
loading validation.2022.data


epoch 1


r.shape = (10000, 19, 19, 31)
nbExamples = 10000
2025-04-02 16:45:30.263628: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-04-02 16:45:30.327775: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node SGD/AssignVariableOp.


79/79 [==============================] - 5s 42ms/step - loss: 6.7136 - policy_loss: 5.8873 - value_loss: 0.6932 - policy_categorical_accuracy: 0.0028 - value_mse: 0.1206
epoch 2


...

r.shape = (10000, 19, 19, 31)
nbExamples = 10000


79/79 [==============================] - 3s 39ms/step - loss: 6.6711 - policy_loss: 5.8780 - value_loss: 0.6930 - policy_categorical_accuracy: 0.0042 - value_mse: 0.1209


r.shape = (10000, 19, 19, 31)
nbExamples = 10000


79/79 [==============================] - 3s 29ms/step - loss: 6.6728 - policy_loss: 5.8813 - value_loss: 0.6930 - policy_categorical_accuracy: 0.0039 - value_mse: 0.1222
val = [6.672774791717529, 5.881328582763672, 0.6929649710655212, 0.0038999998942017555, 0.12224814295768738]


/Users/malikchettih/.pyenv/versions/3.9.21/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
